In [3]:
import pandas as pd

# Function to add IsActualData and DayOfWeek columns to CSV files
def add_columns(csv_file, output_file):
    # Read the CSV file
    df = pd.read_csv(csv_file)
    
    # Add IsActualData column (set to False)
    if 'IsActualData' not in df.columns:
        df['IsActualData'] = False
    
    # Add DayOfWeek column derived from date field
    # First determine which date column to use (StartDate or ReleaseDate)
    date_column = None
    if 'StartDate' in df.columns:
        date_column = 'StartDate'
    elif 'ReleaseDate' in df.columns:
        date_column = 'ReleaseDate'
    elif 'Date' in df.columns:
        date_column = 'Date'
    
    if date_column and 'DayOfWeek' not in df.columns:
        # Handle numeric format like 20250401
        try:
            # First try to convert as is
            df['DayOfWeek'] = pd.to_datetime(df[date_column], errors='coerce').dt.day_name()
            
            # If there are NaN values, try parsing as numeric format
            if df['DayOfWeek'].isna().any():
                # For numeric formats like 20250401
                df['DayOfWeek'] = pd.to_datetime(df[date_column], format='%Y%m%d', errors='coerce').dt.day_name()
        except:
            print(f"Warning: Could not parse dates in {date_column}")
    
    # Save the updated CSV
    df.to_csv(output_file, index=False)
    print(f"Added columns to {output_file}")

# Run for publications and events files
add_columns(r"financial_publications_april_2025.csv", r"publications_updated.csv")
add_columns(r"special_events_april_2025.csv", r"events_updated.csv")

Added columns to publications_updated.csv
Added columns to events_updated.csv


In [5]:
import pandas as pd

# Load the events CSV
events_file = r"events_updated.csv"
output_file = r"events_final.csv"
# Read the file
df = pd.read_csv(events_file)

# Direct approach using ReleaseDate column
date_column = 'ReleaseDate'

# Convert to string first to ensure proper handling
df[date_column] = df[date_column].astype(str)

# For numeric format like 20250401
df['formatted_date'] = df[date_column].apply(lambda x: f"{x[0:4]}-{x[4:6]}-{x[6:8]}" if x.isdigit() and len(x) == 8 else x)

# Now convert to datetime and get day of week
df['DayOfWeek'] = pd.to_datetime(df['formatted_date'], errors='coerce').dt.day_name()

# Drop temporary column
df.drop('formatted_date', axis=1, inplace=True)

# Print a sample to verify
print(df[[date_column, 'DayOfWeek']].head())

# Save the updated CSV
df.to_csv(output_file, index=False)
print(f"Fixed DayOfWeek in {output_file}")

  ReleaseDate DayOfWeek
0    20250401   Tuesday
1    20250401   Tuesday
2    20250401   Tuesday
3    20250401   Tuesday
4    20250401   Tuesday
Fixed DayOfWeek in events_final.csv
